# Figure 2 — Statistical consistency of $\hat{\sigma}$ (`fig:convergence`)

Reproduce the paper figure
*"Convergence of $\hat{\sigma}$ to the true parameter as $n$ increases"*
using the corrected logit-graph pipeline.

## Experiment design (paper Section 4.1.1)

For each configuration we:

1. **Generate** synthetic LG graphs with ground-truth $\sigma \in \{-2,-4,-6,-8\}$,
   neighborhood radius $d \in \{0,1,2\}$, and network size
   $n \in \{10,50,100,200,500,1000,2000\}$.
2. **Estimate** $\hat{\sigma}$ via offset logistic regression
   ($\logit p_{ij} = \sigma + S_i + S_j$, $\beta_1=1$ fixed).
3. **Repeat** the simulate-and-estimate loop for multiple independent replicates
   and report the mean $\hat{\sigma}$ with 95% confidence intervals.

Generation uses Layer-2 Gibbs with `feature_mode="incremental"`;
for $d=0$ the chain reduces to a direct ER sample at $p=\mathrm{expit}(\sigma)$.

## MODE presets

| MODE | `n_values` | reps | `iter_cap` | use case |
|---|---|---:|---:|---|
| `SMOKE` | [50, 100] | 4 | 80k | quick sanity check (~few min) |
| `INSIGHT` | [80, 100] | 2 | 20k | fast bias/variance probe |
| `DEV` | [10, 50, 100, 200] | 8 | 200k | partial convergence curve |
| `PAPER` | [10, …, 2000] | 15 | unlimited | full paper figure (hours) |

Per-cell caches live under `images/correction_paper/sigma_hats_<hash>_*.npz`;
aggregated CSV: `convergence_sigma_<hash>.csv`. Re-runs with the same config
reuse completed cells.

Plot markers are distinct per $\sigma$ (colorblind-friendly; paper review note).

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")

from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from logit_graph.experiments import (
    PRESETS,
    flag_sigma_sweep_issues,
    plot_convergence_sigma,
    run_sigma_sweep,
    summarize_sigma_insights,
)
from logit_graph.experiments.sweeps import _config_hash

MODE = "SMOKE"  # SMOKE | INSIGHT | DEV | PAPER
USE_CACHE = True
N_JOBS = min(4, max(1, (os.cpu_count() or 2) - 1))

OUT_DIR = (Path("..") / ".." / "images" / "correction_paper").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = PRESETS[MODE]["sigma"]
cfg_hash = _config_hash(cfg)
fig_path = OUT_DIR / "convergence_sigma.png"
cache_path = OUT_DIR / f"convergence_sigma_{cfg_hash}.csv"

print(
    f"MODE={MODE}, sigma={cfg.sigma_values}, d={cfg.d_values}, "
    f"n={cfg.n_values}, reps={cfg.n_reps}, iter_cap={cfg.iter_cap}, "
    f"cache={USE_CACHE}, jobs={N_JOBS}"
)
print(f"Config hash: {cfg_hash}")
print(f"Aggregate cache: {cache_path}")

In [ ]:
t0 = time.perf_counter()
df = run_sigma_sweep(cfg, OUT_DIR, use_cache=USE_CACHE, n_jobs=N_JOBS)
elapsed = time.perf_counter() - t0
print(f"Sweep finished in {elapsed:.1f}s — {len(df)} cells, "
      f"{df['n_reps'].iloc[0]} reps/cell")

plot_convergence_sigma(df, fig_path)
print(f"Saved {fig_path}")

display(Image(filename=str(fig_path)))

In [ ]:
# Pivot: mean error (hat_sigma - sigma_true) by (d, sigma, n)
pivot = df.pivot_table(
    index=["d", "sigma_true"],
    columns="n",
    values="sigma_error",
    aggfunc="first",
)
print("Bias (sigma_hat_mean - sigma_true) by cell:")
display(pivot.round(3))

ci_width = df["ci_hi"] - df["ci_lo"]
print(f"\n95% CI half-width at largest n={int(df['n'].max())}:")
at_max = df[df["n"] == df["n"].max()].copy()
at_max["ci_halfwidth"] = (at_max["ci_hi"] - at_max["ci_lo"]) / 2
display(
    at_max[["d", "sigma_true", "sigma_hat_mean", "sigma_error", "ci_halfwidth"]]
    .sort_values(["d", "sigma_true"])
    .reset_index(drop=True)
)

In [ ]:
expected_cells = len(cfg.d_values) * len(cfg.sigma_values) * len(cfg.n_values)
if len(df) < expected_cells:
    print(f"Partial sweep: {len(df)}/{expected_cells} cells (re-run to fill cache).")

try:
    summary = summarize_sigma_insights(df)
except IndexError:
    summary = (
        "Partial sweep — not all (d, sigma, n) cells present at max n; "
        "finish the sweep before summarizing."
    )
print(summary)
(OUT_DIR / "convergence_sigma_insights.txt").write_text(summary + "\n")

issues = flag_sigma_sweep_issues(df, target_density=cfg.target_density)
if len(issues):
    issues_path = OUT_DIR / "convergence_sigma_issues.csv"
    issues.to_csv(issues_path, index=False)
    print(f"\nFlagged {len(issues)} cells -> {issues_path}")
    display(issues[["d", "sigma_true", "n", "sigma_error", "density_mean"]])
else:
    print("\nNo cells flagged for large bias or density drift.")

## Expected qualitative behaviour (paper Section 4.1.1)

- **Small $n$** ($n \lesssim 100$): positive bias and wide 95% CIs.
- **Large $n$**: $\hat{\sigma}$ converges to the true $\sigma$; CIs shrink
  because node-pair observations grow as $O(n^2)$.
- **Across $d$**: convergence rate and accuracy should be broadly comparable
  for $d \in \{0,1,2\}$.

CLI equivalent (headless / batch):
`python notebooks/refactors/run_sigma_experiments.py` with
`LG_EXPERIMENT_MODE=PAPER`.